# COVID-19 Detection from Chest X-Rays
## Using Convolutional Neural Networks

Training a CNN to classify chest X-ray images into two classes:
- **Normal** (healthy lungs)
- **COVID-19** (positive)

Dataset: [COVID-19 Radiography Database](https://www.kaggle.com/datasets/tawsifurrahman/covid19-radiography-database) from Kaggle

We train two architectures and compare:
1. Baseline CNN
2. Deeper CNN with learning rate scheduling

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ReduceLROnPlateau
from sklearn.model_selection import train_test_split

np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow version: {tf.__version__}")

## Data Loading

Load images from the COVID-19 Radiography Database. Update `DATA_PATH` to point to your extracted dataset.

In [ ]:
DATA_PATH = "path/to/COVID-19_Radiography_Dataset"  # <- update this

def load_images(image_paths, size=256, channels=3):
    """Load and preprocess images from file paths."""
    images = []
    for path in image_paths:
        img = tf.keras.preprocessing.image.load_img(path, target_size=(size, size))
        img_array = tf.keras.preprocessing.image.img_to_array(img)
        images.append(img_array)
    return np.array(images) / 255.0  # normalize to [0, 1]

# Load COVID and Normal images
covid_paths = [os.path.join(DATA_PATH, 'COVID', f) for f in os.listdir(os.path.join(DATA_PATH, 'COVID'))[:500]]
normal_paths = [os.path.join(DATA_PATH, 'Normal', f) for f in os.listdir(os.path.join(DATA_PATH, 'Normal'))[:500]]

covid_images = load_images(covid_paths)
normal_images = load_images(normal_paths)

print(f"COVID images:  {covid_images.shape}")
print(f"Normal images: {normal_images.shape}")

## Visualize the Data

In [ ]:
# Show sample images from each class
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for i in range(4):
    axes[0, i].imshow(covid_images[i], cmap='gray')
    axes[0, i].set_title('COVID-19')
    axes[0, i].axis('off')
    axes[1, i].imshow(normal_images[i], cmap='gray')
    axes[1, i].set_title('Normal')
    axes[1, i].axis('off')
plt.tight_layout()
plt.show()

## Prepare Training Data

Combine both classes, create labels (0 = Normal, 1 = COVID), and split into train/test sets.

In [ ]:
images = np.concatenate((normal_images, covid_images), axis=0)
labels = np.concatenate((np.zeros(len(normal_images)), np.ones(len(covid_images))), axis=0)

X_train, X_test, y_train, y_test = train_test_split(
    images, labels, test_size=0.2, random_state=42, stratify=labels
)

print(f"Training:   {X_train.shape[0]} images")
print(f"Test:       {X_test.shape[0]} images")
print(f"Class balance — Normal: {(y_train==0).sum()}, COVID: {(y_train==1).sum()}")

## Data Augmentation

Apply random transformations during training to prevent overfitting and improve generalization.

In [ ]:
datagen = tf.keras.preprocessing.image.ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest'
)

generator = datagen.flow(X_train, y_train, batch_size=8)
print("Data augmentation pipeline ready.")

---
## Model 1: Baseline CNN

In [ ]:
model1 = tf.keras.Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(256, 256, 3)),
    MaxPooling2D((2, 2)),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Conv2D(64, (3, 3), activation='relu'),
    Flatten(),
    Dense(64, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

model1.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model1.summary()

In [ ]:
history1 = model1.fit(
    generator,
    epochs=40,
    validation_data=(X_test, y_test),
    verbose=1
)

In [ ]:
# Training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history1.history['loss'], label='Training Loss')
ax1.plot(history1.history['val_loss'], label='Validation Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Model 1 — Loss')
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot(history1.history['accuracy'], label='Training Accuracy')
ax2.plot(history1.history['val_accuracy'], label='Validation Accuracy')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Model 1 — Accuracy')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

val_acc1 = max(history1.history['val_accuracy'])
print(f"Best validation accuracy (Model 1): {val_acc1:.4f} ({val_acc1*100:.2f}%)")

---
## Model 2: Deeper CNN with LR Scheduling

More layers, batch normalization, and a learning rate scheduler that reduces the LR when validation loss plateaus.

In [ ]:
model2 = tf.keras.Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(256, 256, 3)),
    BatchNormalization(),
    MaxPooling2D((2, 2)),
    Conv2D(64, (3, 3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D((2, 2)),
    Conv2D(128, (3, 3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D((2, 2)),
    Conv2D(128, (3, 3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D((2, 2)),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

model2.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)
model2.summary()

In [ ]:
lr_scheduler = ReduceLROnPlateau(
    monitor='val_loss',
    patience=2,
    factor=0.5,
    min_lr=1e-6,
    verbose=1
)

history2 = model2.fit(
    generator,
    validation_data=(X_test, y_test),
    epochs=50,
    callbacks=[lr_scheduler],
    verbose=1
)

In [ ]:
# Training curves for Model 2
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(history2.history['loss'], label='Training Loss')
axes[0].plot(history2.history['val_loss'], label='Validation Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Model 2 — Loss')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(history2.history['accuracy'], label='Training Accuracy')
axes[1].plot(history2.history['val_accuracy'], label='Validation Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Model 2 — Accuracy')
axes[1].legend()
axes[1].grid(alpha=0.3)

axes[2].plot(history2.history['lr'])
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Learning Rate')
axes[2].set_title('Learning Rate Schedule')
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

val_acc2 = max(history2.history['val_accuracy'])
print(f"Best validation accuracy (Model 2): {val_acc2:.4f} ({val_acc2*100:.2f}%)")

---
## Comparison & Takeaways

- Model 2 (deeper CNN + LR scheduling) typically achieves better validation accuracy
- Data augmentation helps prevent overfitting on this relatively small medical dataset
- The learning rate scheduler visibly kicks in when validation loss plateaus

This isn't a production model — it's a learning exercise in applying CNNs to medical imaging. Real clinical deployment would need much more data, cross-validation, and external validation on unseen hospital data.